# Gold — Cost Trend by Provider

Série temporal de custo mensal por provedor com variação MoM.

**Métricas:**
- `total_cost_usd` — custo total do provedor no mês
- `prev_month_cost_usd` — custo do mês anterior (LAG)
- `mom_variation_pct` — variação Month-over-Month em %

**KPI:** MoM Variation % | **Gráfico:** Tendência por provedor (6 meses)

In [ ]:
# parameters
execution_date = "2025-06-15"
year = 2025
month = 6
day = 15


In [ ]:
import sys
import os

notebooks_root = next(
    (p for p in ["/opt/airflow/notebooks", "/opt/spark/notebooks"]
     if __import__("os").path.exists(p)),
    "/opt/spark/notebooks",
)
if notebooks_root not in sys.path:
    sys.path.insert(0, notebooks_root)

from utils.spark_session import create_spark_session, postgres_jdbc_url, postgres_props
from pyspark.sql import functions as F
from pyspark.sql.types import (
    IntegerType, StringType, DecimalType, TimestampType, DateType
)


In [ ]:
from pyspark.sql.window import Window

spark = create_spark_session("gold_cost_trend_by_provider")

df_entries = spark.read.format("delta").load("s3a://datalake/silver/cost_entries/")

df_monthly = (
    df_entries
    .groupBy("provider", "year", "month")
    .agg(F.round(F.sum("cost_usd"), 4).alias("total_cost_usd"))
)

window_provider = (
    Window.partitionBy("provider")
    .orderBy("year", "month")
)

df_gold = (
    df_monthly
    .withColumn("prev_month_cost_usd", F.lag("total_cost_usd", 1).over(window_provider))
    .withColumn(
        "mom_variation_pct",
        F.round(
            ((F.col("total_cost_usd") - F.col("prev_month_cost_usd"))
             / F.col("prev_month_cost_usd")) * 100,
            2,
        ),
    )
    .withColumn("_processed_at", F.current_timestamp())
    .orderBy("provider", "year", "month")
)

output_path = "s3a://datalake/gold/cost_trend_by_provider/"
df_gold.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(output_path)

print(f"[gold_cost_trend_by_provider] {df_gold.count()} linhas → {output_path}")
df_gold.select("year", "month", "provider", "total_cost_usd", "mom_variation_pct").show(20)
spark.stop()
